In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
catalog_name = "Verse_commonwealth_bank_catalog"

silver_schema = "silver"
gold_schema = "gold"

spark.sql(f"USE CATALOG {catalog_name}")

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {gold_schema}"
)

spark.sql(f"USE SCHEMA {gold_schema}")

print("Gold layer configuration completed")

In [0]:
customers = spark.table(
    "Verse_commonwealth_bank_catalog.silver.customers"
)

accounts = spark.table(
    "Verse_commonwealth_bank_catalog.silver.accounts"
)

transactions = spark.table(
    "Verse_commonwealth_bank_catalog.silver.transactions"
)

loans = spark.table(
    "Verse_commonwealth_bank_catalog.silver.loans"
)

branches = spark.table(
    "Verse_commonwealth_bank_catalog.silver.branches"
)

print("All Silver tables loaded successfully")

In [0]:
def gold_merge(df, table_name, primary_key):

    target_table = f"Verse_commonwealth_bank_catalog.gold.{table_name}"

    # Remove duplicate business keys from source
    df = df.dropDuplicates([primary_key])

    if not spark.catalog.tableExists(target_table):

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )

        print(f"Created Gold table: {target_table}")

    else:

        delta_target = DeltaTable.forName(
            spark,
            target_table
        )

        (
            delta_target.alias("target")
            .merge(
                df.alias("source"),
                f"target.{primary_key} = source.{primary_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"Incremental MERGE completed: {target_table}")

In [0]:
# Aggregate accounts by customer
account_summary = (
    accounts
    .groupBy("customer_id")
    .agg(
        F.countDistinct("account_id").alias("total_accounts"),
        F.sum("current_balance").alias("total_account_balance")
    )
)

# Aggregate loans by customer
loan_summary = (
    loans
    .groupBy("customer_id")
    .agg(
        F.countDistinct("loan_id").alias("total_loans"),
        F.sum("principal_amount").alias("total_loan_amount"),
        F.sum("outstanding_amount").alias("total_outstanding_loan")
    )
)

# Aggregate transactions by customer
# (drop the transactions-side customer_id so customer attribution
# comes from the account owner; this dataset samples them independently)
customer_transactions = (
    transactions
    .drop("customer_id")
    .join(
        accounts.select("account_id", "customer_id"),
        "account_id",
        "inner"
    )
    .filter(F.col("transaction_status") == "SUCCESS")
    .groupBy("customer_id")
    .agg(
        F.countDistinct("transaction_id").alias("total_transactions"),
        F.sum("amount").alias("total_transaction_amount")
    )
)

# Build Customer 360
customer_360 = (
    customers
    .join(account_summary, "customer_id", "left")
    .join(loan_summary, "customer_id", "left")
    .join(customer_transactions, "customer_id", "left")
    .fillna({
        "total_accounts": 0,
        "total_account_balance": 0,
        "total_loans": 0,
        "total_loan_amount": 0,
        "total_outstanding_loan": 0,
        "total_transactions": 0,
        "total_transaction_amount": 0
    })
    .withColumn("gold_updated_at", F.current_timestamp())
)

gold_merge(
    customer_360,
    "gold_customer_360",
    "customer_id"
)

display(customer_360)

In [0]:
account_summary_gold = (
    accounts.alias("a")
    .join(
        customers.select(
            "customer_id",
            "first_name",
            "last_name"
        ).alias("c"),
        "customer_id",
        "left"
    )
    .join(
        branches.select(
            "branch_id",
            "branch_name",
            "city"
        ).alias("b"),
        "branch_id",
        "left"
    )
    .select(
        "account_id",
        "customer_id",
        "first_name",
        "last_name",
        "branch_id",
        "branch_name",
        "city",
        "account_type",
        "currency",
        "current_balance",
        "account_status",
        "open_date"
    )
    .withColumn(
        "gold_updated_at",
        F.current_timestamp()
    )
)

gold_merge(
    account_summary_gold,
    "gold_account_summary",
    "account_id"
)

display(account_summary_gold)

In [0]:
# (drop the transactions-side customer_id so customer attribution
# comes from the account owner; this dataset samples them independently)
transaction_gold = (
    transactions
    .drop("customer_id")
    .alias("t")
    .join(
        accounts.select(
            "account_id",
            "customer_id",
            "branch_id",
            "account_type"
        ).alias("a"),
        "account_id",
        "inner"
    )
    .join(
        customers.select(
            "customer_id",
            "first_name",
            "last_name"
        ).alias("c"),
        "customer_id",
        "inner"
    )
    .withColumn(
        "transaction_date",
        F.to_timestamp("transaction_date")
    )
    .withColumn(
        "transaction_day",
        F.to_date("transaction_date")
    )
    .withColumn(
        "gold_updated_at",
        F.current_timestamp()
    )
    .select(
        "transaction_id",
        "account_id",
        "customer_id",
        "first_name",
        "last_name",
        "branch_id",
        "account_type",
        "transaction_date",
        "transaction_day",
        "transaction_type",
        "amount",
        "currency",
        "channel",
        "transaction_status",
        "gold_updated_at"
    )
)

gold_merge(
    transaction_gold,
    "gold_transaction_details",
    "transaction_id"
)

display(transaction_gold)

In [0]:
loan_gold = (
    loans.alias("l")
    .join(
        customers.select(
            "customer_id",
            "first_name",
            "last_name"
        ).alias("c"),
        "customer_id",
        "inner"
    )
    .join(
        branches.select(
            "branch_id",
            "branch_name",
            "city"
        ).alias("b"),
        "branch_id",
        "left"
    )
    .select(
        "loan_id",
        "customer_id",
        "first_name",
        "last_name",
        "branch_id",
        "branch_name",
        "city",
        "loan_type",
        "principal_amount",
        "interest_rate_pct",
        "tenure_months",
        "outstanding_amount",
        "loan_status",
        "disbursement_date"
    )
    .withColumn(
        "loan_utilization_pct",
        F.when(
            F.col("principal_amount") > 0,
            F.round(
                F.col("outstanding_amount") /
                F.col("principal_amount") * 100,
                2
            )
        ).otherwise(0)
    )
    .withColumn(
        "gold_updated_at",
        F.current_timestamp()
    )
)

gold_merge(
    loan_gold,
    "gold_loan_analytics",
    "loan_id"
)

display(loan_gold)

In [0]:
daily_transaction_summary = (
    transactions
    .withColumn(
        "transaction_day",
        F.to_date("transaction_date")
    )
    .groupBy(
        "transaction_day",
        "transaction_type",
        "transaction_status",
        "currency"
    )
    .agg(
        F.count("transaction_id").alias("transaction_count"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.max("amount").alias("maximum_amount"),
        F.min("amount").alias("minimum_amount")
    )
    .withColumn(
        "gold_updated_at",
        F.current_timestamp()
    )
)

# Composite business key
daily_transaction_summary = daily_transaction_summary.withColumn(
    "summary_key",
    F.sha2(
        F.concat_ws(
            "|",
            F.col("transaction_day").cast("string"),
            F.col("transaction_type"),
            F.col("transaction_status"),
            F.col("currency")
        ),
        256
    )
)

gold_merge(
    daily_transaction_summary,
    "gold_daily_transaction_summary",
    "summary_key"
)

display(daily_transaction_summary)

In [0]:
branch_transactions = (
    transactions.alias("t")
    .join(
        accounts.select(
            "account_id",
            "branch_id"
        ),
        "account_id",
        "inner"
    )
    .filter(
        F.col("transaction_status") == "SETTLED"
    )
    .groupBy("branch_id")
    .agg(
        F.countDistinct("transaction_id").alias(
            "total_transactions"
        ),
        F.sum("amount").alias(
            "total_transaction_amount"
        )
    )
)

branch_accounts = (
    accounts
    .groupBy("branch_id")
    .agg(
        F.countDistinct("account_id").alias(
            "total_accounts"
        ),
        F.sum("current_balance").alias(
            "total_branch_balance"
        )
    )
)

branch_performance = (
    branches
    .join(branch_transactions, "branch_id", "left")
    .join(branch_accounts, "branch_id", "left")
    .fillna({
        "total_transactions": 0,
        "total_transaction_amount": 0,
        "total_accounts": 0,
        "total_branch_balance": 0
    })
    .withColumn(
        "gold_updated_at",
        F.current_timestamp()
    )
)

gold_merge(
    branch_performance,
    "gold_branch_performance",
    "branch_id"
)

display(branch_performance)

In [0]:
banking_kpis = (
    customers.agg(
        F.countDistinct("customer_id").alias(
            "total_customers"
        )
    )
    .crossJoin(
        accounts.agg(
            F.countDistinct("account_id").alias(
                "total_accounts"
            ),
            F.sum("current_balance").alias(
                "total_account_balance"
            )
        )
    )
    .crossJoin(
        transactions.agg(
            F.countDistinct("transaction_id").alias(
                "total_transactions"
            ),
            F.sum(
                F.when(
                    F.col("transaction_status") == "SUCCESS",
                    F.col("amount")
                ).otherwise(0)
            ).alias("total_settled_amount")
        )
    )
    .crossJoin(
        loans.agg(
            F.countDistinct("loan_id").alias(
                "total_loans"
            ),
            F.sum("outstanding_amount").alias(
                "total_outstanding_loans"
            )
        )
    )
    .crossJoin(
        branches.agg(
            F.countDistinct("branch_id").alias(
                "total_branches"
            )
        )
    )
    .withColumn(
        "kpi_run_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "kpi_id",
        F.lit("BANKING_OVERALL")
    )
)

gold_merge(
    banking_kpis,
    "gold_banking_kpis",
    "kpi_id"
)

display(banking_kpis)

In [0]:
gold_tables = [
    "gold_customer_360",
    "gold_account_summary",
    "gold_transaction_details",
    "gold_loan_analytics",
    "gold_daily_transaction_summary",
    "gold_branch_performance",
    "gold_banking_kpis"
]

for table in gold_tables:

    table_name = f"Verse_commonwealth_bank_catalog.gold.{table}"

    print(f"\nChecking: {table_name}")

    df = spark.table(table_name)

    print("Total records:", df.count())

    display(df.limit(5))

In [0]:
%sql
SELECT *
FROM Verse_commonwealth_bank_catalog.gold.gold_customer_360;

In [0]:
%sql
SELECT
    customer_id,
    total_account_balance
FROM Verse_commonwealth_bank_catalog.gold.gold_customer_360
ORDER BY total_account_balance DESC
LIMIT 10;
SELECT *
FROM Verse_commonwealth_bank_catalog.gold.gold_daily_transaction_summary
ORDER BY transaction_day DESC;
SELECT *
FROM Verse_commonwealth_bank_catalog.gold.gold_branch_performance
ORDER BY total_transaction_amount DESC;